In [29]:
mapa_parecer = {
    'veridico': 1,
    'verdadeiro': 1,
    'inveridico': 0,
    'inconclusivo': 2
}

mapa_categoria = {
    'descontextualizacao': 0,
    'fake news': 1,
    'nao se aplica': 2,
    'distorcao de midia': -1,
    'extremismo antidemocratico': 4,
    'discurso de odio': 5
}

In [30]:
import pandas as pd
import unicodedata

result = pd.read_pickle(r"C:\Users\usuario\OneDrive\Documentos\UFG_Remoto\Projects\_ANATEL\TRE\guaia-video-sumarizer-results-generation\result\final_result.pkl")

def normalizar_texto(x):
    x = str(x).strip().lower()
    x = unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode('utf-8')
    return x

def limpar_id(x):
    x = str(x).strip()
    if x.endswith('.mp4'):
        x = x[:-4]
    try:
        return str(int(float(x))) 
    except:
        return x

def normalizar(texto):
    texto = texto.strip().lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto

# Aplicar normalização nas colunas 'parecer' e 'categoria'
result['parecer'] = result['parecer'].apply(normalizar_texto)
result['categoria'] = result['categoria'].apply(normalizar_texto)

result['id'] = result['id'].apply(limpar_id)

result['categoria'] = result['categoria'].apply(lambda x: mapa_categoria.get(normalizar(x), 2))
result['parecer'] = result['parecer'].apply(lambda x: mapa_parecer.get(normalizar(x), 2))
result

,id,link,veridico,inveridico,inconclusivas,categoria,parecer,confianca,justificativa
0,1,https://www.youtube.com/watch?v=tUsAxz2Df44,"""Adriana Acorsi do PT, apoiada pelo Lula. Esse...","""Saiu uma pesquisa recente feita nas capitais ...","""E aí, impressionante, de repente, todos aquel...",0,0,Confiante,O vídeo analisado conta com a exposição de um ...
1,10,https://www.youtube.com/watch?v=H_T5mCdOlrg,"""Qual foi a minha denúncia? O senhor é preside...","""Essa que é a verdade minha gente. Povo de Goi...","""Presidentes de Comissão é que pautam os assun...",0,2,Confiante,O conteúdo elaborado pelo autor do vídeo possu...
2,105,https://www.youtube.com/watch?v=_YR_-s5-xDM,"No vídeo seguinte, uma gravação feita de um ce...","""Mas porquê que o Danilo aqui está com a camis...","""Vai fazer o L pra vocês, agora, neste exato m...",1,0,Confiante,"No conteúdo analisado, a pessoa que elaborou o..."
3,11,https://www.youtube.com/watch?v=ywdoZSCGXDc,"""Aí vem gente fala: “Ah, você criticou ai, tav...","""Por que que querem prender o meu pai? Provave...","""Ah, Bolsonaro caiu dentro da prisão. Ah, poxa...",0,0,Confiante,O autor do vídeo faz uso de uma narrativa cont...
4,111,https://www.youtube.com/watch?v=HmYEpxVjajA,"""Sabe? Os negros não foram libertados pra vira...","""Eles deixaram de ser escravos pra virar vagab...","1% da população que come, que viaja, que vai p...",1,0,Muito confiante,O vídeo elaborado é feito de caso pensado e co...
...,...,...,...,...,...,...,...,...,...
114,201,NaN,NaN,NaN,NaN,5,0,muito confiante,Um suposto vídeo do presidente Lula mostra o p...
115,202,NaN,NaN,NaN,NaN,4,1,muito confiante,Um cidadão comenta uma notícia de que a advoga...
116,203,NaN,NaN,NaN,NaN,5,0,muito confiante,Um cidadão comenta em um vídeo sobre um supost...
117,204,NaN,NaN,NaN,NaN,4,1,muito confiante,Um jornal notícia a prisão do mecânico do 8 de...


In [32]:
df_real = result.copy()

In [33]:
df_real.groupby('parecer')['id'].nunique()

parecer
0    61
1    27
2    31
Name: id, dtype: int64

In [34]:
df_real.groupby('categoria')['id'].nunique()

categoria
-1     4
 0    27
 1    21
 2    27
 4    22
 5    18
Name: id, dtype: int64

In [35]:
# Dicionário de mudanças
mudancas = {
    "126": [0, 5],
    "138": [1, 2],
    "135": [0, 1],
    "30": [0, 5],
    "202": [0, 5],
    "151": [0, 1],
    "127": [0, 4],
    "197": [1, 2],
    "193": [1, 2],
    "174": [0, 0],
    "200": [1, 2],
    "204": [1, 2],
    "145": [1, 0],
    "147": [1, 0],
    "153": [1, 0],
    "195": [1, 2],
    "199": [1, 2],
    "160": [2, 0],
    "162": [0, 0],
    "183": [1, 2],
    "192": [2, 2],
    "22": [0, 0],
    "191": [1, 2],
    "144": [1, 0],
    "36": [1, 2],
    "196": [0, 0],
    "33": [0, 4],
    "171": [0, 0],
    "35": [1, 2],
    "158": [1, 0],
    "132": [0, 0],
    "156": [0, 5],
    "23": [1, 2],
    "13": [0, 4],
    "133": [0, 0],
    "34": [0, 4],
    "27": [0, 2],
    "18":[0, 0],
    "160": [0, 0],
}

# Garante que video_id é string
df_real['id'] = df_real['id'].astype(str)

# Aplica e imprime mudanças
for id, (novo_parecer, nova_categoria) in mudancas.items():
    linhas_antes = df_real[df_real['id'] == id][['id', 'parecer', 'categoria']]
    
    df_real.loc[df_real['id'] == id, 'parecer'] = novo_parecer
    df_real.loc[df_real['id'] == id, 'categoria'] = nova_categoria
    
    linhas_depois = df_real[df_real['id'] == id][['id', 'parecer', 'categoria']]
    
    print(f"➡️ Alteração para id = {id}")
    print("Antes:")
    print(linhas_antes.to_string(index=False))
    print("Depois:")
    print(linhas_depois.to_string(index=False))
    print("-" * 50)


➡️ Alteração para id = 126
Antes:
 id  parecer  categoria
126        0         -1
Depois:
 id  parecer  categoria
126        0          5
--------------------------------------------------
➡️ Alteração para id = 138
Antes:
 id  parecer  categoria
138        0         -1
Depois:
 id  parecer  categoria
138        1          2
--------------------------------------------------
➡️ Alteração para id = 135
Antes:
 id  parecer  categoria
135        0         -1
Depois:
 id  parecer  categoria
135        0          1
--------------------------------------------------
➡️ Alteração para id = 30
Antes:
id  parecer  categoria
30        0         -1
Depois:
id  parecer  categoria
30        0          5
--------------------------------------------------
➡️ Alteração para id = 202
Antes:
 id  parecer  categoria
202        1          4
Depois:
 id  parecer  categoria
202        0          5
--------------------------------------------------
➡️ Alteração para id = 151
Antes:
 id  parecer  categoria
15

In [36]:
df_real.groupby('parecer')['id'].nunique()

parecer
0    70
1    31
2    18
Name: id, dtype: int64

In [55]:
df_real.groupby('categoria')['id'].nunique()

categoria
descontextualizacao           35
discurso de odio              20
extremismo antidemocratico    12
fake news                     20
nao se aplica                 32
Name: id, dtype: int64

In [51]:
# Dicionários de mapeamento
mapa_parecer = {
    'veridico': 1,
    'inveridico': 0,
    'inconclusivo': 2
}

mapa_categoria = {
    'descontextualizacao': 0,
    'fake news': 1,
    'nao se aplica': 2,
    'distorcao de midia': -1,
    'extremismo antidemocratico': 4,
    'discurso de odio': 5
}

# Inverte os dicionários
mapa_parecer_inv = {v: k for k, v in mapa_parecer.items()}
mapa_categoria_inv = {v: k for k, v in mapa_categoria.items()}

# Aplica os mapeamentos inversos
df_real['parecer'] = df_real['parecer'].map(mapa_parecer_inv)
df_real['categoria'] = df_real['categoria'].map(mapa_categoria_inv)

df_real

,id,link,veridico,inveridico,inconclusivas,categoria,parecer,confianca,justificativa
0,1,https://www.youtube.com/watch?v=tUsAxz2Df44,"""Adriana Acorsi do PT, apoiada pelo Lula. Esse...","""Saiu uma pesquisa recente feita nas capitais ...","""E aí, impressionante, de repente, todos aquel...",descontextualizacao,inveridico,Confiante,O vídeo analisado conta com a exposição de um ...
1,10,https://www.youtube.com/watch?v=H_T5mCdOlrg,"""Qual foi a minha denúncia? O senhor é preside...","""Essa que é a verdade minha gente. Povo de Goi...","""Presidentes de Comissão é que pautam os assun...",descontextualizacao,inconclusivo,Confiante,O conteúdo elaborado pelo autor do vídeo possu...
2,105,https://www.youtube.com/watch?v=_YR_-s5-xDM,"No vídeo seguinte, uma gravação feita de um ce...","""Mas porquê que o Danilo aqui está com a camis...","""Vai fazer o L pra vocês, agora, neste exato m...",fake news,inveridico,Confiante,"No conteúdo analisado, a pessoa que elaborou o..."
3,11,https://www.youtube.com/watch?v=ywdoZSCGXDc,"""Aí vem gente fala: “Ah, você criticou ai, tav...","""Por que que querem prender o meu pai? Provave...","""Ah, Bolsonaro caiu dentro da prisão. Ah, poxa...",descontextualizacao,inveridico,Confiante,O autor do vídeo faz uso de uma narrativa cont...
4,111,https://www.youtube.com/watch?v=HmYEpxVjajA,"""Sabe? Os negros não foram libertados pra vira...","""Eles deixaram de ser escravos pra virar vagab...","1% da população que come, que viaja, que vai p...",fake news,inveridico,Muito confiante,O vídeo elaborado é feito de caso pensado e co...
...,...,...,...,...,...,...,...,...,...
114,201,NaN,NaN,NaN,NaN,discurso de odio,inveridico,muito confiante,Um suposto vídeo do presidente Lula mostra o p...
115,202,NaN,NaN,NaN,NaN,discurso de odio,inveridico,muito confiante,Um cidadão comenta uma notícia de que a advoga...
116,203,NaN,NaN,NaN,NaN,discurso de odio,inveridico,muito confiante,Um cidadão comenta em um vídeo sobre um supost...
117,204,NaN,NaN,NaN,NaN,nao se aplica,veridico,muito confiante,Um jornal notícia a prisão do mecânico do 8 de...


In [52]:
result.groupby('parecer')['video_id'].nunique()

parecer
0    61
1    27
2    31
Name: video_id, dtype: int64

In [57]:
df_real.groupby('categoria')['id'].nunique()

categoria
descontextualizacao           35
discurso de odio              20
extremismo antidemocratico    12
fake news                     20
nao se aplica                 32
Name: id, dtype: int64

In [ ]:
df_real.to_pickle(r"result\final_result_curadoria.pkl")